# Скачивание ВСЕГО архива аудио из Telegram-канала

Вход по вашему аккаунту (а не по боту). Читает всю историю канала и
скачивает все аудио с их названиями — ничего пересылать вручную не нужно.

**Что нужно заранее:** `api_id` и `api_hash` с https://my.telegram.org
(раздел «API development tools»).

Запускайте ячейки по порядку кнопкой ▶.

## 1. Установка

In [ ]:
!pip install -q telethon

## 2. Ваши данные
Заполните и запустите ячейку.

In [ ]:
API_ID   = 1234567                 # число с my.telegram.org
API_HASH = "ваш_api_hash"         # строка с my.telegram.org
PHONE    = "+7XXXXXXXXXX"         # ваш номер телефона в Telegram
CHANNEL  = "@ilmdinislam"         # канал-источник (@username или ссылка t.me/...)
LIMIT    = 0                        # 0 = все сообщения; или число последних

## 3. Вход и скачивание
Запустите ячейку. При первом запуске Telegram пришлёт **код** в
приложение — появится поле для ввода прямо под ячейкой, введите код туда.
Если включена двухэтапная защита — затем попросит пароль.

In [ ]:
import json, re
from pathlib import Path
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeAudio, DocumentAttributeFilename

OUT = Path('audio'); OUT.mkdir(exist_ok=True)
manifest_path = OUT / 'manifest.json'
manifest = json.loads(manifest_path.read_text('utf-8')) if manifest_path.exists() else []
done = {m['message_id'] for m in manifest}
used = {m['file'] for m in manifest}

def clean(s):
    s = re.sub(r'[\\/:*?"<>|\n\r\t]+', ' ', (s or '').strip())
    return (re.sub(r'\s+', ' ', s).strip()[:150]) or 'audio'

def attrs(doc):
    a = fn = None
    for x in doc.attributes:
        if isinstance(x, DocumentAttributeAudio): a = x
        elif isinstance(x, DocumentAttributeFilename): fn = x.file_name
    return a, fn

client = TelegramClient('user_session', API_ID, API_HASH)

async def main():
    await client.start(phone=PHONE)
    ent = await client.get_entity(CHANNEL)
    print('Канал:', getattr(ent, 'title', CHANNEL))
    n = 0
    async for msg in client.iter_messages(ent, limit=(LIMIT or None)):
        doc = getattr(msg, 'document', None)
        if not doc: continue
        a, fn = attrs(doc)
        if not (a or (doc.mime_type and doc.mime_type.startswith('audio'))): continue
        if msg.id in done: continue
        if a and not a.voice and (a.title or a.performer):
            title = ' - '.join(p for p in [(a.performer or '').strip(), (a.title or '').strip()] if p)
        elif fn:
            title = Path(fn).stem
        else:
            title = (msg.message or '').splitlines()[0] if msg.message else f'audio_{msg.id}'
        ext = (Path(fn).suffix if fn and '.' in fn else ('.ogg' if a and a.voice else '.mp3'))
        name = clean(title) + ext; i = 2
        while name in used or (OUT / name).exists():
            name = f'{clean(title)} ({i}){ext}'; i += 1
        used.add(name)
        print('↓', name)
        await client.download_media(msg, file=str(OUT / name))
        manifest.append({'message_id': msg.id, 'title': title, 'file': name,
                         'duration_sec': getattr(a, 'duration', None),
                         'date': msg.date.isoformat() if msg.date else None})
        done.add(msg.id); n += 1
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), 'utf-8')
    print(f'\nГотово. Скачано новых: {n}. Всего в manifest: {len(manifest)}')

await main()

## 4. Скачать всё себе одним архивом

In [ ]:
import shutil
shutil.make_archive('audio_export', 'zip', 'audio')
from google.colab import files
files.download('audio_export.zip')